# 05 - Masked Thermal Reconstruction Training

**專案**: 熱成像超解析度 + 去背重建  
**目標**: 訓練 U-Net 模型，將低解析度 IR (32x24) 還原為去背、保留溫度的高解析度影像 (256x256)

## Pipeline
1. 讀取 IR `. npy` 並正規化
2. 使用 Affine Matrix 將 IR 對齊到 RGB 座標系
3.  讀取 LabelMe JSON 產生 Binary Mask
4.  合成 Target = Warped_IR * Mask
5. 裁切 256x256
6. 訓練 U-Net

---
## Step 0: 環境設定

In [ ]:
import os
import json
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from datetime import datetime
from typing import List, Dict, Tuple, Optional
import warnings
warnings. filterwarnings('ignore')

# 檢查 GPU
device = torch.device('cuda' if torch. cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## Step 1: 路徑與參數配置

In [ ]:
# ========================================
# 路徑配置
# ========================================
BASE_DIR = Path(os.getcwd())
OUTPUT_DIR = BASE_DIR / 'outputgary'

# 來源資料
THERMAL_DIR = OUTPUT_DIR / 'aligned_dataset' / 'thermal'  # . npy 檔案
ANNOTATION_DIR = OUTPUT_DIR / 'labelme_project' / 'annotations'  # .json 檔案
IMAGE_DIR = OUTPUT_DIR / 'labelme_project' / 'images'  # RGB 參考圖 (用於取得尺寸)

# 輸出目錄
DATASET_DIR = OUTPUT_DIR / 'masked_thermal_dataset'
INPUT_DIR = DATASET_DIR / 'input_ir'
TARGET_DIR = DATASET_DIR / 'target_mask'
MODEL_DIR = OUTPUT_DIR / 'models'

for d in [INPUT_DIR, TARGET_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ========================================
# 核心對齊參數 (已計算完成)
# ========================================
# Affine Matrix: [Scale, 0, Offset_X], [0, Scale, Offset_Y]
AFFINE_MATRIX = np.array([
    [54.8231, 0.0000, -366.5609],
    [0.0000, 54.8231, -325.8295]
], dtype=np.float32)

# RGB 尺寸
RGB_WIDTH = 960
RGB_HEIGHT = 720

# IR 原始尺寸
IR_WIDTH = 32
IR_HEIGHT = 24

# 訓練裁切尺寸
CROP_SIZE = 256

# ========================================
# 訓練參數
# ========================================
BATCH_SIZE = 8
NUM_EPOCHS = 150
LEARNING_RATE = 1e-4
NUM_WORKERS = 4

print(f"📁 資料路徑:")
print(f"   Thermal NPY: {THERMAL_DIR}")
print(f"   Annotations: {ANNOTATION_DIR}")
print(f"   RGB Images: {IMAGE_DIR}")
print(f"\n⚙️ Affine Matrix:")
print(f"   Scale: {AFFINE_MATRIX[0, 0]:.4f}")
print(f"   Offset X: {AFFINE_MATRIX[0, 2]:.4f}")
print(f"   Offset Y: {AFFINE_MATRIX[1, 2]:.4f}")

---
## Step 2: 資料處理函數

In [ ]:
# ========================================
# Step 2: 資料處理函數 (修正版 - 含正方形處理)
# ========================================

def normalize_ir(ir_data: np.ndarray) -> Tuple[np.ndarray, float, float]:
    """
    正規化 IR 數值到 0~1，並回傳 min/max 供還原
    """
    ir_min = float(ir_data.min())
    ir_max = float(ir_data.max())
    
    if ir_max - ir_min < 1e-6:
        return np.zeros_like(ir_data, dtype=np.float32), ir_min, ir_max
    
    normalized = (ir_data - ir_min) / (ir_max - ir_min)
    return normalized. astype(np.float32), ir_min, ir_max


def load_labelme_annotation(json_path: Path, img_height: int, img_width: int) -> Tuple[np.ndarray, str]:
    """
    從 LabelMe JSON 檔案產生 Binary Mask 和姿勢標籤
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    mask = np.zeros((img_height, img_width), dtype=np.uint8)
    pose_label = 'unknown'
    
    valid_labels = ['lying', 'sitting', 'standing', 'fallen', 'right_lying', 'left_lying']
    
    for shape in data. get('shapes', []):
        if shape['shape_type'] == 'polygon':
            label = shape. get('label', '')
            if label in valid_labels:
                points = np.array(shape['points'], dtype=np.int32)
                cv2.fillPoly(mask, [points], 1)
                pose_label = label
    
    return mask, pose_label


def warp_ir_to_rgb(ir_image: np.ndarray, affine_matrix: np.ndarray,
                   output_size: Tuple[int, int] = (RGB_WIDTH, RGB_HEIGHT)) -> np. ndarray:
    """
    使用 Affine Matrix 將 IR (32x24) 對齊到 RGB 座標系 (960x720)
    """
    warped = cv2.warpAffine(
        ir_image,
        affine_matrix,
        output_size,
        flags=cv2.INTER_LINEAR,
        borderMode=cv2. BORDER_CONSTANT,
        borderValue=0
    )
    return warped


def pad_to_square(image: np.ndarray, pad_value: float = 0) -> Tuple[np.ndarray, dict]:
    """
    將影像 padding 成正方形（保持長寬比）
    
    Returns:
        padded: 正方形影像
        pad_info: padding 資訊，供還原使用
    """
    h, w = image.shape[:2]
    max_side = max(h, w)
    
    # 計算 padding
    pad_h = max_side - h
    pad_w = max_side - w
    
    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left
    
    # Padding
    if len(image.shape) == 2:
        padded = np. pad(image, ((pad_top, pad_bottom), (pad_left, pad_right)), 
                       mode='constant', constant_values=pad_value)
    else:
        padded = np.pad(image, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), 
                       mode='constant', constant_values=pad_value)
    
    pad_info = {
        'original_h': h,
        'original_w': w,
        'pad_top': pad_top,
        'pad_bottom': pad_bottom,
        'pad_left': pad_left,
        'pad_right': pad_right
    }
    
    return padded, pad_info

---
## Step 3: 產生訓練資料集

In [ ]:
# ========================================
# Step 3: 產生訓練資料集 (修正版 - 正方形處理)
# ========================================

POSE_LABELS = {
    'standing': 0,
    'sitting': 1,
    'lying': 2,
    'right_lying': 3,
    'left_lying': 4,
    'fallen': 5,
    'unknown': 6
}

def generate_training_data():
    """
    產生訓練資料:
    
    Input:  原始 IR (32x24) → pad to square (32x32) → resize (256x256)
    Target: Warped IR × Mask (960x720) → pad to square (960x960) → resize (256x256)
    
    這樣保持長寬比，不會變形！
    """
    
    import re
    
    def extract_number(filename: str) -> Optional[str]:
        match = re.search(r'(\d{5})', filename)
        return match. group(1) if match else None
    
    # 取得所有檔案
    npy_files = sorted(THERMAL_DIR.glob('*.npy'))
    json_files = sorted(ANNOTATION_DIR.glob('*.json'))
    
    print(f"找到 {len(npy_files)} 個 thermal npy 檔案")
    print(f"找到 {len(json_files)} 個 annotation json 檔案")
    
    # 建立序號對應字典
    npy_dict = {extract_number(f.stem): f for f in npy_files if extract_number(f.stem)}
    json_dict = {extract_number(f.stem): f for f in json_files if extract_number(f.stem)}
    
    common_keys = set(npy_dict.keys()) & set(json_dict.keys())
    print(f"找到 {len(common_keys)} 對配對資料")
    
    if len(common_keys) == 0:
        print("\n❌ 沒有找到配對資料！")
        return []
    
    successful = 0
    failed = 0
    skipped_empty_mask = 0
    pose_counts = {label: 0 for label in POSE_LABELS.keys()}
    data_pairs = []
    
    for key in tqdm(sorted(common_keys), desc="產生訓練資料"):
        try:
            npy_path = npy_dict[key]
            json_path = json_dict[key]
            
            # ===== 1. 讀取原始 IR (32x24) =====
            ir_raw = np.load(npy_path). astype(np.float32)
            
            # ===== 2. 正規化 =====
            ir_normalized, ir_min, ir_max = normalize_ir(ir_raw)
            
            # ===== 3. 讀取 LabelMe mask 和姿勢標籤 =====
            mask, pose_label = load_labelme_annotation(json_path, RGB_HEIGHT, RGB_WIDTH)
            
            if mask.sum() < 100:
                skipped_empty_mask += 1
                continue
            
            # ===== 4.  Warp IR 到 RGB 座標系 (960x720) =====
            ir_warped = warp_ir_to_rgb(ir_normalized, AFFINE_MATRIX)
            
            # ===== 5. 去背：Warped IR × Mask =====
            ir_masked = ir_warped * mask. astype(np.float32)
            
            # ===== 6.  Pad to Square 再 Resize =====
            
            # Input: 原始 IR (24x32) → pad to (32x32) → resize (256x256)
            input_padded, input_pad_info = pad_to_square(ir_normalized, pad_value=0)
            input_resized = cv2.resize(input_padded, (CROP_SIZE, CROP_SIZE),
                                       interpolation=cv2.INTER_LINEAR)
            
            # Target: 去背 IR (720x960) → pad to (960x960) → resize (256x256)
            target_padded, target_pad_info = pad_to_square(ir_masked, pad_value=0)
            target_resized = cv2.resize(target_padded, (CROP_SIZE, CROP_SIZE),
                                        interpolation=cv2.INTER_LINEAR)
            
            # ===== 7. 儲存 =====
            output_key = f"sample_{key}"
            input_path = INPUT_DIR / f"{output_key}.npy"
            target_path = TARGET_DIR / f"{output_key}.npy"
            
            np.save(input_path, input_resized)
            np.save(target_path, target_resized)
            
            data_pairs.append({
                'key': output_key,
                'input_path': str(input_path),
                'target_path': str(target_path),
                'pose_label': pose_label,
                'pose_id': POSE_LABELS. get(pose_label, 6),
                'ir_min': ir_min,
                'ir_max': ir_max,
                'input_pad_info': input_pad_info,
                'target_pad_info': target_pad_info,
                'original_npy': str(npy_path),
                'original_json': str(json_path)
            })
            
            pose_counts[pose_label] += 1
            successful += 1
            
        except Exception as e:
            failed += 1
            if failed <= 5:
                print(f"\n⚠️ 處理 {key} 失敗: {e}")
    
    print(f"\n✅ 完成！")
    print(f"   成功: {successful}")
    print(f"   失敗: {failed}")
    print(f"   跳過 (空 mask): {skipped_empty_mask}")
    print(f"\n📊 姿勢分佈:")
    for label, count in pose_counts.items():
        if count > 0:
            print(f"   {label}: {count}")
    
    # 儲存配對資訊
    pairs_path = DATASET_DIR / 'data_pairs.json'
    with open(pairs_path, 'w') as f:
        json.dump(data_pairs, f, indent=2)
    print(f"\n📄 配對資訊已儲存到: {pairs_path}")
    
    return data_pairs

# 執行
data_pairs = generate_training_data()

---
## Step 4: 視覺化驗證

In [ ]:
# ========================================
# Step 4: 視覺化驗證 (報告版)
# ========================================

def visualize_samples_for_report(data_pairs: List[Dict], num_samples: int = 6):
    """
    視覺化訓練資料樣本 (適合報告)
    
    左: 原始 IR (32x24)
    中: Input - Resize 後的 IR (256x256)
    右: Target - Resize 後的 IR × Mask (去背, 256x256)
    """
    if not data_pairs:
        print("沒有資料可視覺化")
        return
    
    # 隨機選取不同姿勢的樣本
    np.random.seed(42)  # 固定種子，方便重現
    indices = np.random.choice(len(data_pairs), min(num_samples, len(data_pairs)), replace=False)
    
    fig, axes = plt.subplots(len(indices), 3, figsize=(15, 4 * len(indices)))
    
    # 設定標題字體大小
    title_fontsize = 12
    
    for i, idx in enumerate(indices):
        pair = data_pairs[idx]
        
        # ===== 1. 讀取原始 IR (32x24) =====
        npy_path = Path(pair['original_npy'])
        ir_raw = np.load(npy_path)  # (24, 32)
        
        # ===== 2. 讀取 Input 和 Target =====
        input_img = np.load(pair['input_path'])   # (256, 256), 0~1
        target_img = np.load(pair['target_path']) # (256, 256), 0~1 去背
        
        # ===== 3. 繪製 =====
        
        # 左: 原始 IR (32x24)
        im0 = axes[i, 0].imshow(ir_raw, cmap='jet')
        axes[i, 0].set_title(f"Original IR (32×24)\n{pair['key']}", fontsize=title_fontsize)
        axes[i, 0].axis('off')
        
        # 中: Input - Resize 後的 IR (256x256)
        im1 = axes[i, 1].imshow(input_img, cmap='jet', vmin=0, vmax=1)
        axes[i, 1]. set_title(f"Input: IR Resized (256×256)\nPose: {pair['pose_label']}", fontsize=title_fontsize)
        axes[i, 1].axis('off')
        
        # 右: Target - Resize 後的 IR × Mask (去背, 256x256)
        im2 = axes[i, 2]. imshow(target_img, cmap='jet', vmin=0, vmax=1)
        axes[i, 2].set_title(f"Target: IR × Mask (256×256)\nBackground Removed", fontsize=title_fontsize)
        axes[i, 2].axis('off')
    
    # 加入 colorbar
    fig.subplots_adjust(right=0.92)
    cbar_ax = fig.add_axes([0.94, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(im2, cax=cbar_ax)
    cbar.set_label('Normalized Temperature (0~1)', fontsize=11)
    
    plt.suptitle('Training Data: IR Super-Resolution + Background Removal', 
                 fontsize=14, fontweight='bold', y=1.01)
    
    plt.tight_layout()
    plt.savefig(DATASET_DIR / 'sample_visualization_report.png', dpi=200, bbox_inches='tight')
    plt.show()
    
    print(f"\n📊 視覺化已儲存到: {DATASET_DIR / 'sample_visualization_report.png'}")


# 執行視覺化
visualize_samples_for_report(data_pairs, num_samples=6)

---
## Step 5: PyTorch Dataset

In [ ]:
# ========================================
# Step 5: PyTorch Dataset (修正版 - 資料已經是 0~1)
# ========================================

class MaskedThermalDataset(Dataset):
    """
    Masked Thermal 訓練資料集
    """
    
    def __init__(self, data_pairs: List[Dict], augment: bool = True):
        self.data_pairs = data_pairs
        self.augment = augment
    
    def __len__(self):
        return len(self.data_pairs)
    
    def __getitem__(self, idx):
        pair = self.data_pairs[idx]
        
        # 載入資料 (已經是 0~1 範圍)
        input_img = np.load(pair['input_path']).astype(np.float32)
        target_img = np.load(pair['target_path']).astype(np.float32)
        
        # Data Augmentation
        if self.augment:
            # 隨機水平翻轉
            if np.random.random() > 0.5:
                input_img = np.fliplr(input_img). copy()
                target_img = np.fliplr(target_img).copy()
            
            # 隨機垂直翻轉
            if np.random.random() > 0.5:
                input_img = np.flipud(input_img).copy()
                target_img = np.flipud(target_img).copy()
        
        # 轉換為 tensor (C, H, W)
        input_tensor = torch.from_numpy(input_img).unsqueeze(0)
        target_tensor = torch. from_numpy(target_img). unsqueeze(0)
        
        return input_tensor, target_tensor


# 分割訓練/驗證集
def split_dataset(data_pairs: List[Dict], val_ratio: float = 0.2):
    np.random.seed(42)
    indices = np.random.permutation(len(data_pairs))
    val_size = int(len(data_pairs) * val_ratio)
    
    val_indices = indices[:val_size]
    train_indices = indices[val_size:]
    
    train_pairs = [data_pairs[i] for i in train_indices]
    val_pairs = [data_pairs[i] for i in val_indices]
    
    return train_pairs, val_pairs

# 建立 DataLoader
if data_pairs:
    train_pairs, val_pairs = split_dataset(data_pairs)
    
    train_dataset = MaskedThermalDataset(train_pairs, augment=True)
    val_dataset = MaskedThermalDataset(val_pairs, augment=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    print(f"\n📊 資料集統計:")
    print(f"   訓練集: {len(train_dataset)} 筆")
    print(f"   驗證集: {len(val_dataset)} 筆")
    print(f"   Batch size: {BATCH_SIZE}")
    print(f"   訓練 batches/epoch: {len(train_loader)}")

---
## Step 6: U-Net 模型定義

In [ ]:
# ========================================
# Step 6: U-Net 模型定義 (完整版)
# ========================================

class DoubleConv(nn.Module):
    """(Conv => BN => ReLU) * 2"""
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn. Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn. Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self. double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn. MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    
    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn. ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x1, x2):
        x1 = self.up(x1)
        
        # Padding if needed
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = nn.functional.pad(x1, [diffX // 2, diffX - diffX // 2,
                                     diffY // 2, diffY - diffY // 2])
        
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    """
    U-Net for IR Super-Resolution + Background Removal
    
    Input:  1 channel (IR heatmap, 0~1)
    Output: 1 channel (Masked high-res IR, 0~1)
    """
    
    def __init__(self, in_channels=1, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        
        self.inc = DoubleConv(in_channels, features[0])
        self.down1 = Down(features[0], features[1])
        self. down2 = Down(features[1], features[2])
        self.down3 = Down(features[2], features[3])
        self.down4 = Down(features[3], features[3] * 2)
        
        self.up1 = Up(features[3] * 2, features[3])
        self.up2 = Up(features[3], features[2])
        self.up3 = Up(features[2], features[1])
        self.up4 = Up(features[1], features[0])
        
        self.outc = nn.Conv2d(features[0], out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self. up4(x, x1)
        
        x = self. outc(x)
        return self.sigmoid(x)


# 建立模型
model = UNet(in_channels=1, out_channels=1). to(device)

# 計算參數量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p. numel() for p in model. parameters() if p.requires_grad)

print(f"\n🧠 U-Net 模型:")
print(f"   總參數量: {total_params:,}")
print(f"   可訓練參數: {trainable_params:,}")

---
## Step 7: 訓練

In [ ]:
# ========================================
# Step 7: 訓練 (L1 Loss + AdamW + CosineAnnealing)
# ========================================

def train_model(model, train_loader, val_loader, num_epochs, learning_rate, val_every=5):
    """
    訓練 U-Net (IR 超解析度 + 去背)
    
    使用 CosineAnnealingLR: LR 從初始值平滑下降到 eta_min
    """
    
    criterion = nn.L1Loss()
    optimizer = optim. AdamW(model. parameters(), lr=learning_rate, weight_decay=1e-4)
    
    # CosineAnnealingLR: 平滑下降學習率
    # T_max: 週期長度 (設為總 epoch 數，整個訓練過程平滑下降一次)
    # eta_min: 最小學習率
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=num_epochs, 
        eta_min=1e-6
    )
    
    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    best_val_loss = float('inf')
    best_epoch = 0
    
    print(f"\n{'='*70}")
    print(f"開始訓練 (IR Super-Resolution + Background Removal)")
    print(f"Loss: L1Loss")
    print(f"Optimizer: AdamW (weight_decay=1e-4)")
    print(f"Scheduler: CosineAnnealingLR (eta_min=1e-6)")
    print(f"Epochs: {num_epochs}")
    print(f"Learning Rate: {learning_rate} -> 1e-6")
    print(f"Device: {device}")
    print(f"Validation: Every {val_every} epochs")
    print(f"{'='*70}")
    
    for epoch in range(num_epochs):
        # ==================== Training ====================
        model.train()
        train_loss = 0.0
        
        for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        history['train_loss'].append(train_loss)
        
        # ==================== Scheduler Step (每個 epoch 都要呼叫) ====================
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        history['lr'].append(current_lr)
        
        # ==================== Validation ====================
        do_val = (epoch + 1) % val_every == 0 or epoch == 0 or epoch == num_epochs - 1
        
        if do_val:
            model. eval()
            val_loss = 0.0
            
            with torch.no_grad():
                for inputs, targets in val_loader:
                    inputs = inputs.to(device)
                    targets = targets.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
                    val_loss += loss. item()
            
            val_loss /= len(val_loader)
            history['val_loss'].append(val_loss)
            
            # 儲存最佳模型
            save_marker = ''
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch + 1
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'val_loss': val_loss,
                }, MODEL_DIR / 'best_model.pth')
                save_marker = ' *'
            
            lr_str = f"{current_lr:.2e}"
            val_str = f"{val_loss:.6f}"
            best_str = f"{best_val_loss:.6f}"
            print(f"Epoch {epoch+1:3d}/{num_epochs} | Train: {train_loss:.6f} | Val: {val_str} | LR: {lr_str} | Best: {best_str}{save_marker}")
        else:
            lr_str = f"{current_lr:.2e}"
            print(f"Epoch {epoch+1:3d}/{num_epochs} | Train: {train_loss:. 6f} | LR: {lr_str}")
    
    print(f"\n{'='*70}")
    print(f"訓練完成!")
    best_str = f"{best_val_loss:.6f}"
    print(f"Best Val Loss: {best_str} (Epoch {best_epoch})")
    print(f"模型已儲存到: {MODEL_DIR / 'best_model.pth'}")
    print(f"{'='*70}")
    
    # 儲存最終模型
    torch.save(model.state_dict(), MODEL_DIR / 'final_model. pth')
    
    # 儲存訓練歷史
    with open(MODEL_DIR / 'training_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    
    return history


# ========================================
# 重新初始化模型 (重要！)
# ========================================
model = UNet(in_channels=1, out_channels=1). to(device)

print(f"模型已重新初始化")
print(f"總參數量: {sum(p.numel() for p in model.parameters()):,}")


# ========================================
# 開始訓練
# ========================================
history = train_model(model, train_loader, val_loader, NUM_EPOCHS, LEARNING_RATE, val_every=5)

---
## Step 8: 訓練曲線視覺化

In [ ]:
# ========================================
# Step 8: 訓練曲線視覺化 (修正版)
# ========================================

def plot_training_history(history: Dict, val_every: int = 5):
    """
    繪製訓練曲線 (處理不同頻率的 validation)
    
    Args:
        history: 包含 train_loss, val_loss, lr 的字典
        val_every: 每幾個 epoch 做一次 validation
    """
    
    train_loss = history['train_loss']
    val_loss = history['val_loss']
    lr = history['lr']
    
    num_epochs = len(train_loss)
    
    # 計算 validation 的 epoch 位置
    # 第 1 個 epoch (index 0) + 每 val_every 個 epoch + 最後一個 epoch
    val_epochs = [0]  # 第 1 個 epoch
    for i in range(val_every - 1, num_epochs, val_every):
        if i not in val_epochs:
            val_epochs.append(i)
    if num_epochs - 1 not in val_epochs:
        val_epochs.append(num_epochs - 1)  # 最後一個 epoch
    
    # 確保 val_epochs 數量與 val_loss 一致
    val_epochs = val_epochs[:len(val_loss)]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # ===== 1. Loss Curve =====
    epochs = range(1, num_epochs + 1)
    val_epoch_display = [e + 1 for e in val_epochs]  # 轉換為 1-based
    
    axes[0]. plot(epochs, train_loss, 'b-', label='Train Loss', linewidth=2, alpha=0.8)
    axes[0].plot(val_epoch_display, val_loss, 'r-o', label='Val Loss', linewidth=2, markersize=4)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('L1 Loss', fontsize=12)
    axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim([1, num_epochs])
    
    # 標註最佳 val_loss
    best_val_idx = np.argmin(val_loss)
    best_val_epoch = val_epoch_display[best_val_idx]
    best_val_value = val_loss[best_val_idx]
    axes[0].annotate(f'Best: {best_val_value:.6f}\n(Epoch {best_val_epoch})',
                     xy=(best_val_epoch, best_val_value),
                     xytext=(best_val_epoch + 10, best_val_value + 0.02),
                     fontsize=10,
                     arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
                     color='red')
    
    # ===== 2. Loss Curve (Log Scale) =====
    axes[1].semilogy(epochs, train_loss, 'b-', label='Train Loss', linewidth=2, alpha=0.8)
    axes[1].semilogy(val_epoch_display, val_loss, 'r-o', label='Val Loss', linewidth=2, markersize=4)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('L1 Loss (log scale)', fontsize=12)
    axes[1].set_title('Loss Curve (Log Scale)', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3, which='both')
    axes[1].set_xlim([1, num_epochs])
    
    # ===== 3. Learning Rate =====
    axes[2].plot(epochs, lr, 'g-', linewidth=2)
    axes[2].set_xlabel('Epoch', fontsize=12)
    axes[2]. set_ylabel('Learning Rate', fontsize=12)
    axes[2].set_title('Learning Rate Schedule (CosineAnnealing)', fontsize=14, fontweight='bold')
    axes[2].set_yscale('log')
    axes[2].grid(True, alpha=0.3)
    axes[2].set_xlim([1, num_epochs])
    
    # 標註 LR 範圍
    axes[2].axhline(y=lr[0], color='gray', linestyle='--', alpha=0.5)
    axes[2].axhline(y=lr[-1], color='gray', linestyle='--', alpha=0.5)
    axes[2].text(num_epochs * 0.7, lr[0] * 1.2, f'Initial: {lr[0]:.2e}', fontsize=10, color='gray')
    axes[2].text(num_epochs * 0.7, lr[-1] * 0.5, f'Final: {lr[-1]:.2e}', fontsize=10, color='gray')
    
    plt.suptitle('Training History - IR Super-Resolution + Background Removal', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.savefig(MODEL_DIR / 'training_curves.png', dpi=200, bbox_inches='tight')
    plt.show()
    
    # ===== 印出統計資訊 =====
    print(f"\n{'='*60}")
    print(f"📊 訓練統計")
    print(f"{'='*60}")
    print(f"總 Epochs: {num_epochs}")
    print(f"最終 Train Loss: {train_loss[-1]:.6f}")
    print(f"最終 Val Loss: {val_loss[-1]:.6f}")
    print(f"最佳 Val Loss: {best_val_value:.6f} (Epoch {best_val_epoch})")
    print(f"Learning Rate: {lr[0]:.2e} → {lr[-1]:.2e}")
    print(f"{'='*60}")


# 執行繪圖
plot_training_history(history, val_every=5)

---
## Step 9: 推論測試

---
## 完成！

### 輸出檔案:
- `outputgary/masked_thermal_dataset/` - 訓練資料
  - `input_ir/` - 輸入 IR (256x256)
  - `target_mask/` - 目標 (去背 IR, 256x256)
- `outputgary/models/` - 模型檔案
  - `best_model.pth` - 最佳模型
  - `final_model.pth` - 最終模型
  - `training_history.json` - 訓練歷史
  - `training_curves.png` - 訓練曲線
  - `inference_results.png` - 推論結果

In [ ]:
# ========================================
# Step 8: 訓練曲線視覺化 (從 JSON 讀取)
# ========================================

import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 讀取訓練歷史
MODEL_DIR = Path('outputgary/models')  # 根據您的路徑調整
history_path = MODEL_DIR / 'training_history.json'

with open(history_path, 'r') as f:
    history = json.load(f)

print(f"✅ 已讀取訓練歷史: {history_path}")
print(f"   Train Loss: {len(history['train_loss'])} 個點")
print(f"   Val Loss: {len(history['val_loss'])} 個點")
print(f"   LR: {len(history['lr'])} 個點")


def plot_training_history(history: dict, val_every: int = 5):
    """
    繪製訓練曲線 (處理不同頻率的 validation)
    """
    
    train_loss = history['train_loss']
    val_loss = history['val_loss']
    lr = history['lr']
    
    num_epochs = len(train_loss)
    
    # 計算 validation 的 epoch 位置
    # 第 1 個 epoch (index 0) + 每 val_every 個 epoch + 最後一個 epoch
    val_epochs = [1]  # 第 1 個 epoch (1-based)
    for i in range(val_every, num_epochs + 1, val_every):
        if i not in val_epochs:
            val_epochs.append(i)
    if num_epochs not in val_epochs:
        val_epochs.append(num_epochs)  # 最後一個 epoch
    
    # 確保 val_epochs 數量與 val_loss 一致
    val_epochs = val_epochs[:len(val_loss)]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # ===== 1. Loss Curve =====
    epochs = range(1, num_epochs + 1)
    
    axes[0]. plot(epochs, train_loss, 'b-', label='Train Loss', linewidth=2, alpha=0.8)
    axes[0].plot(val_epochs, val_loss, 'r-o', label='Val Loss', linewidth=2, markersize=4)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('L1 Loss', fontsize=12)
    axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim([1, num_epochs])
    
    # 標註最佳 val_loss
    best_val_idx = np.argmin(val_loss)
    best_val_epoch = val_epochs[best_val_idx]
    best_val_value = val_loss[best_val_idx]
    axes[0].annotate(f'Best: {best_val_value:.6f}\n(Epoch {best_val_epoch})',
                     xy=(best_val_epoch, best_val_value),
                     xytext=(best_val_epoch - 30, best_val_value + 0.03),
                     fontsize=10,
                     arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
                     color='red')
    
    # ===== 2. Loss Curve (Log Scale) =====
    axes[1].semilogy(epochs, train_loss, 'b-', label='Train Loss', linewidth=2, alpha=0.8)
    axes[1].semilogy(val_epochs, val_loss, 'r-o', label='Val Loss', linewidth=2, markersize=4)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('L1 Loss (log scale)', fontsize=12)
    axes[1].set_title('Loss Curve (Log Scale)', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3, which='both')
    axes[1]. set_xlim([1, num_epochs])
    
    # ===== 3. Learning Rate =====
    axes[2]. plot(epochs, lr, 'g-', linewidth=2)
    axes[2].set_xlabel('Epoch', fontsize=12)
    axes[2]. set_ylabel('Learning Rate', fontsize=12)
    axes[2].set_title('Learning Rate Schedule (CosineAnnealing)', fontsize=14, fontweight='bold')
    axes[2].set_yscale('log')
    axes[2].grid(True, alpha=0.3)
    axes[2].set_xlim([1, num_epochs])
    
    # 標註 LR 範圍
    axes[2].axhline(y=lr[0], color='gray', linestyle='--', alpha=0.5)
    axes[2].axhline(y=lr[-1], color='gray', linestyle='--', alpha=0.5)
    axes[2].text(num_epochs * 0.02, lr[0] * 1.3, f'Initial: {lr[0]:.2e}', fontsize=10, color='gray')
    axes[2].text(num_epochs * 0.02, lr[-1] * 0.4, f'Final: {lr[-1]:.2e}', fontsize=10, color='gray')
    
    plt.suptitle('Training History - IR Super-Resolution + Background Removal', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    plt. tight_layout()
    plt. savefig(MODEL_DIR / 'training_curves.png', dpi=200, bbox_inches='tight')
    plt.show()
    
    # ===== 印出統計資訊 =====
    print(f"\n{'='*60}")
    print(f"📊 訓練統計")
    print(f"{'='*60}")
    print(f"總 Epochs: {num_epochs}")
    print(f"最終 Train Loss: {train_loss[-1]:.6f}")
    print(f"最終 Val Loss: {val_loss[-1]:.6f}")
    print(f"最佳 Val Loss: {best_val_value:.6f} (Epoch {best_val_epoch})")
    print(f"Learning Rate: {lr[0]:.2e} → {lr[-1]:.2e}")
    print(f"Train/Val Gap: {val_loss[-1] - train_loss[-1]:.6f}")
    print(f"{'='*60}")


# 執行繪圖
plot_training_history(history, val_every=5)

In [ ]:
# ========================================
# Step 9: 推論測試 (含原始 IR 版)
# ========================================

import os
import json
import re
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict

# ========================================
# 1. 路徑設定
# ========================================
BASE_DIR = Path(os.getcwd())
OUTPUT_DIR = BASE_DIR / 'outputgary'
DATASET_DIR = OUTPUT_DIR / 'masked_thermal_dataset'
INPUT_DIR = DATASET_DIR / 'input_ir'
TARGET_DIR = DATASET_DIR / 'target_mask'
MODEL_DIR = OUTPUT_DIR / 'models'
ANNOTATION_DIR = OUTPUT_DIR / 'labelme_project' / 'annotations'
THERMAL_DIR = OUTPUT_DIR / 'aligned_dataset' / 'thermal'  # 原始 IR 路徑

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ========================================
# 2. 從標註檔案讀取姿勢
# ========================================
def get_pose_from_annotation(json_path: Path) -> str:
    """
    從 LabelMe JSON 檔案讀取姿勢標籤
    """
    valid_labels = ['lying', 'sitting', 'standing', 'fallen', 'right_lying', 'left_lying']
    
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        for shape in data. get('shapes', []):
            label = shape. get('label', '')
            if label in valid_labels:
                return label
    except:
        pass
    
    return 'unknown'


def build_data_pairs_with_pose_and_original():
    """
    從資料夾建立配對，包含姿勢和原始 IR 路徑
    """
    input_files = sorted(INPUT_DIR.glob('*.npy'))
    
    print(f"📁 Input IR 檔案數: {len(input_files)}")
    print(f"📁 Annotation 檔案數: {len(list(ANNOTATION_DIR.glob('*.json')))}")
    print(f"📁 Original Thermal 檔案數: {len(list(THERMAL_DIR.glob('*.npy')))}")
    
    data_pairs = []
    pose_counts = {}
    
    for input_path in input_files:
        target_path = TARGET_DIR / input_path.name
        
        if not target_path.exists():
            continue
        
        # 從檔名提取序號 (sample_00000 -> 00000)
        match = re.search(r'(\d{5})', input_path.stem)
        if not match:
            continue
        
        frame_num = match.group(1)
        
        # 找對應的標註檔案 (frame_00000. json)
        json_path = ANNOTATION_DIR / f'frame_{frame_num}.json'
        
        # 找原始 IR 檔案 (thermal_00000.npy 或 frame_00000.npy)
        original_ir_path = None
        for pattern in [f'thermal_{frame_num}. npy', f'frame_{frame_num}.npy', f'{frame_num}.npy']:
            candidate = THERMAL_DIR / pattern
            if candidate.exists():
                original_ir_path = candidate
                break
        
        # 如果找不到，嘗試搜尋
        if original_ir_path is None:
            candidates = list(THERMAL_DIR. glob(f'*{frame_num}*.npy'))
            if candidates:
                original_ir_path = candidates[0]
        
        if original_ir_path is None:
            continue
        
        if json_path.exists():
            pose_label = get_pose_from_annotation(json_path)
        else:
            pose_label = 'unknown'
        
        data_pairs.append({
            'key': input_path.stem,
            'input_path': str(input_path),
            'target_path': str(target_path),
            'original_ir_path': str(original_ir_path),
            'pose_label': pose_label,
            'frame_num': frame_num
        })
        
        pose_counts[pose_label] = pose_counts.get(pose_label, 0) + 1
    
    print(f"\n✅ 成功配對: {len(data_pairs)} 筆")
    print(f"\n📊 姿勢分佈:")
    for pose, count in sorted(pose_counts.items(), key=lambda x: -x[1]):
        print(f"   {pose}: {count}")
    
    return data_pairs, pose_counts


data_pairs, pose_counts = build_data_pairs_with_pose_and_original()

# ========================================
# 3. 定義 U-Net 模型
# ========================================
class DoubleConv(nn. Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self. double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    
    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2. size()[3] - x1. size()[3]
        x1 = nn.functional.pad(x1, [diffX // 2, diffX - diffX // 2,
                                     diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.inc = DoubleConv(in_channels, features[0])
        self.down1 = Down(features[0], features[1])
        self.down2 = Down(features[1], features[2])
        self.down3 = Down(features[2], features[3])
        self.down4 = Down(features[3], features[3] * 2)
        
        self.up1 = Up(features[3] * 2, features[3])
        self.up2 = Up(features[3], features[2])
        self.up3 = Up(features[2], features[1])
        self.up4 = Up(features[1], features[0])
        
        self.outc = nn.Conv2d(features[0], out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self. up4(x, x1)
        
        x = self. outc(x)
        return self.sigmoid(x)


# ========================================
# 4.  載入模型
# ========================================
model = UNet(in_channels=1, out_channels=1). to(device)

checkpoint_path = MODEL_DIR / 'best_model.pth'
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"\n✅ 載入模型: {checkpoint_path}")
print(f"   Epoch: {checkpoint['epoch'] + 1}")
print(f"   Val Loss: {checkpoint['val_loss']:.6f}")

# ========================================
# 5. 按姿勢分類推論 (每種姿勢 3 張，含原始 IR)
# ========================================
def inference_by_pose_with_original(model, data_pairs: List[Dict], samples_per_pose: int = 3):
    """
    每種姿勢各顯示指定數量的推論結果
    
    顯示順序: 原始 IR (32×24) | Input (256×256) | Target (GT) | Prediction
    """
    if not data_pairs:
        print("❌ data_pairs 是空的")
        return
    
    model.eval()
    
    # 按姿勢分組
    pose_groups = {}
    for pair in data_pairs:
        pose = pair['pose_label']
        if pose == 'unknown':
            continue
        if pose not in pose_groups:
            pose_groups[pose] = []
        pose_groups[pose].append(pair)
    
    poses = sorted(pose_groups.keys())
    num_poses = len(poses)
    
    print(f"\n📊 有效姿勢類別: {poses}")
    
    # 計算總行數
    total_rows = num_poses * samples_per_pose
    
    fig, axes = plt.subplots(total_rows, 4, figsize=(16, 3.5*total_rows))
    
    if total_rows == 1:
        axes = axes.reshape(1, -1)
    
    row = 0
    pose_mae = {pose: [] for pose in poses}
    last_im_temp = None
    
    # 設定欄位標題 (只在第一行顯示)
    col_titles = ['Original IR\n(32×24)', 'Input\n(256×256)', 'Target (GT)\n(256×256)', 'Prediction\n(256×256)']
    
    with torch.no_grad():
        for pose_idx, pose in enumerate(poses):
            samples = pose_groups[pose]
            np. random.seed(42)
            selected = np.random.choice(
                len(samples), 
                min(samples_per_pose, len(samples)), 
                replace=False
            )
            
            for j, idx in enumerate(selected):
                pair = samples[idx]
                
                # 載入原始 IR (32×24)
                original_ir = np.load(pair['original_ir_path']).astype(np.float32)
                
                # 正規化原始 IR (與訓練時一致)
                ir_min, ir_max = original_ir. min(), original_ir.max()
                if ir_max - ir_min > 1e-6:
                    original_ir_norm = (original_ir - ir_min) / (ir_max - ir_min)
                else:
                    original_ir_norm = np.zeros_like(original_ir)
                
                # 載入 Input 和 Target
                input_img = np.load(pair['input_path']).astype(np. float32)
                target_img = np.load(pair['target_path']).astype(np.float32)
                
                # 推論
                input_tensor = torch.from_numpy(input_img).unsqueeze(0).unsqueeze(0).to(device)
                output = model(input_tensor)
                output_img = output.squeeze(). cpu().numpy()
                
                # 計算 MAE
                mae = np. abs(target_img - output_img).mean()
                pose_mae[pose].append(mae)
                
                # ===== 繪製 =====
                
                # 第一欄: 原始 IR (32×24)
                axes[row, 0].imshow(original_ir_norm, cmap='jet', vmin=0, vmax=1)
                axes[row, 0].set_title(f"Original IR (32×24)\n{pair['key']}", fontsize=9)
                axes[row, 0].axis('off')
                
                # 第二欄: Input (256×256)
                axes[row, 1].imshow(input_img, cmap='jet', vmin=0, vmax=1)
                axes[row, 1].set_title(f"Input (256×256)", fontsize=9)
                axes[row, 1].axis('off')
                
                # 第三欄: Target (GT)
                axes[row, 2].imshow(target_img, cmap='jet', vmin=0, vmax=1)
                axes[row, 2].set_title(f"Target (GT)\nPose: {pose}", fontsize=9, fontweight='bold')
                axes[row, 2].axis('off')
                
                # 第四欄: Prediction
                last_im_temp = axes[row, 3].imshow(output_img, cmap='jet', vmin=0, vmax=1)
                axes[row, 3].set_title(f"Prediction\nMAE: {mae:.4f}", fontsize=9)
                axes[row, 3].axis('off')
                
                # 在每個姿勢的第一行左側加上姿勢標籤
                if j == 0:
                    axes[row, 0].set_ylabel(pose. upper(), fontsize=11, fontweight='bold', 
                                            rotation=0, labelpad=60, va='center')
                
                row += 1
    
    # Colorbar
    fig.subplots_adjust(right=0.93)
    
    if last_im_temp is not None:
        cbar_ax = fig.add_axes([0.95, 0.15, 0.015, 0.7])
        cbar = fig.colorbar(last_im_temp, cax=cbar_ax)
        cbar.set_label('Normalized Temperature (0~1)', fontsize=10)
    
    # 計算總平均 MAE
    all_mae = [m for maes in pose_mae.values() for m in maes]
    avg_mae = np.mean(all_mae)
    
    plt.suptitle(f'IR Super-Resolution + Background Removal Results\n'
                 f'Original (32×24) → Input (256×256) → Prediction (256×256) | Average MAE: {avg_mae:.4f}', 
                 fontsize=14, fontweight='bold', y=1.01)
    
    plt.tight_layout()
    plt.savefig(MODEL_DIR / 'inference_by_pose.png', dpi=200, bbox_inches='tight')
    plt.show()
    
    # 印出統計
    print(f"\n{'='*60}")
    print(f"📊 各姿勢 MAE 統計")
    print(f"{'='*60}")
    for pose in poses:
        if pose_mae[pose]:
            avg = np.mean(pose_mae[pose])
            print(f"   {pose:12s}: MAE = {avg:.4f} (n={len(pose_mae[pose])})")
    print(f"{'='*60}")
    print(f"   {'Overall':12s}: MAE = {avg_mae:.4f}")
    print(f"{'='*60}")
    print(f"\n✅ 圖片已儲存: {MODEL_DIR / 'inference_by_pose.png'}")


# ========================================
# 6. 執行推論
# ========================================
if len(data_pairs) > 0:
    inference_by_pose_with_original(model, data_pairs, samples_per_pose=3)
else:
    print("❌ 沒有找到配對資料")